In [1]:
import pandas as pd
import numpy as np
import numba

from numba import jit
from tqdm.auto import tqdm

In [2]:
tqdm.pandas()

In [3]:
dataset_path = "D:/Repo/epi-thesis/dataset/"
working_path = "D:/Repo/epi-thesis/workflow/03. Encoding New Dataset, XGBoost/"

In [4]:
def load_histone_data(file_name, type):
    columns = ['chrom', 'chromStart', 'chromEnd', 'name']
    df = pd.read_csv(file_name, sep="\t", header=None, names=columns)
    df["type"] = type
    return df

In [5]:
def encode_histone(row, histone_df, threshold = 0.8, histone_length = 146):
    tss = row["tss"]

    hist_df = histone_df.loc[# Inside the +/- 2k from TSS
                             (((histone_df["chromStart"] >= tss - 2000) & (histone_df["chromEnd"] <= tss + 2000)) |
                             # Intersect with -2k or +2 from TSS
                             (((histone_df["chromEnd"] - (tss - 2000))/histone_length).between(threshold, 1.0)) |
                             ((((tss + 2000) - histone_df["chromStart"])/histone_length).between(threshold, 1.0)))]
    
    hist_count_dict = hist_df["type"].value_counts().to_dict()
    hist_count_dict.update({'histone_count_total': len(hist_df.index)})

# Load Data

## Load Gene Expression

In [6]:
gene_df = pd.read_csv(f"{dataset_path}histone_count_overlap80.csv", sep="\t")
display(gene_df.head())
display(gene_df.shape)

,h_chrom,tss_start,tss_end,h_chromStart,h_chromEnd,h_gene,h_test_id,h_gene_id,h_locus,h_sample_1,...,n_cdsStartStat,n_cdsEndStat,n_exonFrames,tss,H3K4me3,H3K9ac,H3K9me3,H3K27ac,H3K27me3,histone_total_count
0,chr1,63418,67418,69090,70008,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,...,cmpl,cmpl,"-1,0,0,",65418,0,0,0,0,0,0
1,chr1,321891,325891,323891,328581,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,...,none,none,"-1,-1,-1,",323891,0,0,0,0,0,0
2,chr1,365658,369658,367658,368597,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,...,cmpl,cmpl,"0,",367658,0,0,0,0,0,0
3,chr1,760970,764970,761585,794889,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,...,none,none,"-1,-1,-1,-1,",762970,0,0,0,0,0,0
4,chr1,760970,764970,761585,794889,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,...,none,none,"-1,-1,-1,-1,",762970,0,0,0,0,0,0


(59366, 42)

In [29]:
gene_exp_df = gene_df[['h_chrom', 'h_chromStart', 'h_chromEnd', 'h_value_1', 'n_strand', 'tss', 'tss_start', 'tss_end']]
display(gene_exp_df.head())
display(gene_exp_df.shape)

,h_chrom,h_chromStart,h_chromEnd,h_value_1,n_strand,tss,tss_start,tss_end
0,chr1,69090,70008,0.000000,+,65418,63418,67418
1,chr1,323891,328581,0.047392,+,323891,321891,325891
2,chr1,367658,368597,0.000000,+,367658,365658,369658
3,chr1,761585,794889,2.468570,+,762970,760970,764970
4,chr1,761585,794889,2.468570,+,762970,760970,764970


(59366, 8)

## Load Histone

In [30]:
h3k4me3_df = load_histone_data(f"{dataset_path}HepG2_Male.histone.H3K4me3.peak.bed", "h3k4me3")
h3k9ac_df = load_histone_data(f"{dataset_path}HepG2_Male.histone.H3K9ac.peak.bed", "h3k9ac")
h3k9me3_df = load_histone_data(f"{dataset_path}HepG2_Male.histone.H3K9me3.peak.bed", "h3k9me3")
h3k27ac_df = load_histone_data(f"{dataset_path}HepG2_Male.histone.H3K27ac.peak.bed", "h3k27ac")
h3k27me3_df = load_histone_data(f"{dataset_path}HepG2_Male.histone.H3K27me3.peak.bed", "h3k27me3")

In [9]:
display(h3k4me3_df.head())

,chrom,chromStart,chromEnd,name,type
0,chr10,119808,119954,chr10_173,h3k4me3
1,chr10,119956,120102,chr10_174,h3k4me3
2,chr10,122100,122246,chr10_185,h3k4me3
3,chr10,122308,122454,chr10_186,h3k4me3
4,chr10,180346,180492,chr10_489,h3k4me3


# Data Encoding

In [40]:
def encode_histone(row, histone_df, threshold = 0.8, histone_length = 146):  
  chrom = row["h_chrom"]
  tss_start = row['tss_start']
  tss_end = row['tss_end']

  hist_df = histone_df.loc[(histone_df['chrom'] == chrom) & 
                            (
                                # Inside the +/- 2KB from TSS
                                ((histone_df['chromStart'] >= tss_start) & (histone_df['chromEnd'] <= tss_end)) |

                                # Overlap at the start
                                ((histone_df['chromStart'] < tss_start) & 
                                (histone_df['chromEnd'] > tss_start) & 
                                (histone_df['chromEnd'] < tss_end) & 
                                ((histone_df['chromEnd'] - tss_start)/ histone_length >= threshold)) |

                                # Overlap at the end
                                ((histone_df['chromStart'] > tss_start) &
                                (histone_df['chromStart'] < tss_end) &
                                (histone_df['chromEnd'] > tss_end) &
                                ((tss_end - histone_df['chromStart'])/histone_length >= threshold))
                            )
                          ]

  start_idx = hist_df[['chromStart']].to_numpy()
  arr = np.zeros(4000)
  idx = [x for x in start_idx - tss_start]
  arr[idx] = 1

  return arr

The histone: 

- h3k4me3_df
- h3k9ac_df
- h3k9me3_df
- h3k27ac_df
- h3k27me3_df

In [41]:
gene_exp_df.loc[:, 'h3k4me3'] = gene_exp_df.progress_apply(lambda row: encode_histone(row, h3k4me3_df), axis = 1)

  0%|          | 0/59366 [00:00<?, ?it/s]

In [43]:
gene_exp_df.loc[:, 'h3k9ac'] = gene_exp_df.progress_apply(lambda row: encode_histone(row, h3k9ac_df), axis = 1)

  0%|          | 0/59366 [00:00<?, ?it/s]

C:\Users\abdul\AppData\Local\Temp\ipykernel_4124\186883512.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gene_exp_df.loc[:, 'h3k9ac'] = gene_exp_df.progress_apply(lambda row: encode_histone(row, h3k9ac_df), axis = 1)


In [44]:
gene_exp_df.loc[:, 'h3k9me3'] = gene_exp_df.progress_apply(lambda row: encode_histone(row, h3k9me3_df), axis = 1)

  0%|          | 0/59366 [00:00<?, ?it/s]

C:\Users\abdul\AppData\Local\Temp\ipykernel_4124\2034792428.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gene_exp_df.loc[:, 'h3k9me3'] = gene_exp_df.progress_apply(lambda row: encode_histone(row, h3k9me3_df), axis = 1)


In [45]:
gene_exp_df.loc[:, 'h3k27ac'] = gene_exp_df.progress_apply(lambda row: encode_histone(row, h3k27ac_df), axis = 1)

  0%|          | 0/59366 [00:00<?, ?it/s]

C:\Users\abdul\AppData\Local\Temp\ipykernel_4124\3060020973.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gene_exp_df.loc[:, 'h3k27ac'] = gene_exp_df.progress_apply(lambda row: encode_histone(row, h3k27ac_df), axis = 1)


In [46]:
gene_exp_df.loc[:, 'h3k27me3'] = gene_exp_df.progress_apply(lambda row: encode_histone(row, h3k27me3_df), axis = 1)

  0%|          | 0/59366 [00:00<?, ?it/s]

C:\Users\abdul\AppData\Local\Temp\ipykernel_4124\1384121094.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gene_exp_df.loc[:, 'h3k27me3'] = gene_exp_df.progress_apply(lambda row: encode_histone(row, h3k27me3_df), axis = 1)


In [47]:
display(gene_exp_df.head())
display(gene_exp_df.shape)

,h_chrom,h_chromStart,h_chromEnd,h_value_1,n_strand,tss,tss_start,tss_end,h3k4me3,h3k9ac,h3k9me3,h3k27ac,h3k27me3
0,chr1,69090,70008,0.000000,+,65418,63418,67418,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,chr1,323891,328581,0.047392,+,323891,321891,325891,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,chr1,367658,368597,0.000000,+,367658,365658,369658,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,chr1,761585,794889,2.468570,+,762970,760970,764970,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,chr1,761585,794889,2.468570,+,762970,760970,764970,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


(59366, 13)

## Saving into CSV Files

In [48]:
# Convert array values to string
# df['column2'] = df['column2'].apply(lambda x: ','.join(map(str, x)))
# h3k4me3
# h3k9ac
# h3k9me3
# h3k27ac
# h3k27me3

gene_exp_df['h3k4me3']  = gene_exp_df['h3k4me3'].apply(lambda x: ','.join(map(str, x)))
gene_exp_df['h3k9ac']   = gene_exp_df['h3k9ac'].apply(lambda x: ','.join(map(str, x)))
gene_exp_df['h3k9me3']  = gene_exp_df['h3k9me3'].apply(lambda x: ','.join(map(str, x)))
gene_exp_df['h3k27ac']  = gene_exp_df['h3k27ac'].apply(lambda x: ','.join(map(str, x)))
gene_exp_df['h3k27me3'] = gene_exp_df['h3k27me3'].apply(lambda x: ','.join(map(str, x)))

C:\Users\abdul\AppData\Local\Temp\ipykernel_4124\2572685829.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gene_exp_df['h3k4me3']  = gene_exp_df['h3k4me3'].apply(lambda x: ','.join(map(str, x)))
C:\Users\abdul\AppData\Local\Temp\ipykernel_4124\2572685829.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gene_exp_df['h3k9ac']   = gene_exp_df['h3k9ac'].apply(lambda x: ','.join(map(str, x)))
C:\Users\abdul\AppData\Local\Temp\ipykernel_4124\2572685829.py:11: SettingWithCopyWarning: 
A value is trying to

In [49]:
# Save to CSV
gene_exp_df.to_csv('output.csv', index=False)

In [55]:
# Read the CSV file
mylist = []

for chunk in pd.read_csv('output.csv', chunksize=20000):
    mylist.append(chunk)

full_data = pd.concat(mylist, axis = 0)
del mylist

In [56]:
full_data.head()

,h_chrom,h_chromStart,h_chromEnd,h_value_1,n_strand,tss,tss_start,tss_end,h3k4me3,h3k9ac,h3k9me3,h3k27ac,h3k27me3
0,chr1,69090,70008,0.000000,+,65418,63418,67418,"0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0...."
1,chr1,323891,328581,0.047392,+,323891,321891,325891,"0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0...."
2,chr1,367658,368597,0.000000,+,367658,365658,369658,"0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0...."
3,chr1,761585,794889,2.468570,+,762970,760970,764970,"0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0...."
4,chr1,761585,794889,2.468570,+,762970,760970,764970,"0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0....","0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0...."


In [58]:
# Convert string representation back to arrays
full_data['h3k4me3']  = full_data['h3k4me3'].apply(lambda x: np.array(list(map(float, x.split(',')))))
full_data['h3k9ac']   = full_data['h3k9ac'].apply(lambda x: np.array(list(map(float, x.split(',')))))
full_data['h3k9me3']  = full_data['h3k9me3'].apply(lambda x: np.array(list(map(float, x.split(',')))))
full_data['h3k27ac']  = full_data['h3k27ac'].apply(lambda x: np.array(list(map(float, x.split(',')))))
full_data['h3k27me3'] = full_data['h3k27me3'].apply(lambda x: np.array(list(map(float, x.split(',')))))

In [59]:
display(full_data.head())
display(full_data.shape)

,h_chrom,h_chromStart,h_chromEnd,h_value_1,n_strand,tss,tss_start,tss_end,h3k4me3,h3k9ac,h3k9me3,h3k27ac,h3k27me3
0,chr1,69090,70008,0.000000,+,65418,63418,67418,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,chr1,323891,328581,0.047392,+,323891,321891,325891,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,chr1,367658,368597,0.000000,+,367658,365658,369658,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,chr1,761585,794889,2.468570,+,762970,760970,764970,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,chr1,761585,794889,2.468570,+,762970,760970,764970,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


(59366, 13)